# 03 — Foundational Analysis

Run the five core tests on the EO operator system:
1. **Completeness** — Does every verb map to at least one operator?
2. **Minimality** — Is each operator necessary?
3. **Orthogonality** — Are the nine operators semantically distinct?
4. **Clustering** — Does k=9 emerge from unsupervised analysis?
5. **Boundaries** — Which verbs sit between two operators?

**Requires:** Pre-computed data OR embeddings from 02_Embeddings.

In [ ]:
# ═══ Load pre-computed analysis results ═══
import json, numpy as np
from pyodide.http import pyfetch
from collections import Counter

HELIX = ['NUL','DES','INS','SEG','CON','SYN','ALT','SUP','REC']

# Load pre-computed results
metrics = None
boundaries = None
confusion = None

for name, var in [('metrics', 'metrics'), ('boundaries', 'boundaries'), ('confusion', 'confusion')]:
    try:
        resp = await pyfetch(f'../data/{name}.json')
        data = json.loads(await resp.string())
        exec(f'{var} = data')
        print(f'  Loaded {name}.json')
    except:
        print(f'  {name}.json not found')

print('\nData loaded. Run individual test cells below.')

## Test 1: Completeness

In [ ]:
# ═══ Completeness: Operator distribution ═══
if metrics and 'distribution' in metrics:
    dist = metrics['distribution']
    total = sum(dist.values())
    print(f'Total verbs assigned: {total:,}')
    print(f'\n{"Operator":>8s} {"Count":>8s} {"Pct":>7s}  Bar')
    print('-' * 60)
    for op in HELIX:
        count = dist.get(op, 0)
        pct = count / total * 100 if total > 0 else 0
        bar = '█' * int(pct * 2)
        print(f'{op:>8s} {count:>8,d} {pct:>6.1f}%  {bar}')
    
    # Check for gaps
    min_count = min(dist.get(op, 0) for op in HELIX)
    max_count = max(dist.get(op, 0) for op in HELIX)
    ratio = max_count / min_count if min_count > 0 else float('inf')
    print(f'\nImbalance ratio: {ratio:.1f}x (max/min)')
    if ratio > 10:
        print('  WARNING: Large imbalance may indicate operator scope issues')
    else:
        print('  Reasonable balance across operators')
else:
    print('No pre-computed metrics. Generate embeddings and run analysis.')

## Test 3: Orthogonality

In [ ]:
# ═══ Orthogonality: Inter-operator confusion ═══
if confusion and 'confusion' in confusion:
    conf = confusion['confusion']
    print('Operator Confusion Matrix (embedding-based):')
    print(f'{"":>5s}', end='')
    for op in HELIX:
        print(f'{op:>6s}', end='')
    print()
    print('-' * (5 + 6*9))
    
    for from_op in HELIX:
        print(f'{from_op:>5s}', end='')
        for to_op in HELIX:
            val = conf.get(from_op, {}).get(to_op, 0)
            if from_op == to_op:
                print(f'  ----', end='')
            elif val > 50:
                print(f'{val:>6d}', end='')  # High confusion
            else:
                print(f'{val:>6d}', end='')
        print()
    
    # Find highest confusion pairs
    pairs = []
    for a in HELIX:
        for b in HELIX:
            if a < b:
                total = conf.get(a, {}).get(b, 0) + conf.get(b, {}).get(a, 0)
                pairs.append((a, b, total))
    pairs.sort(key=lambda x: -x[2])
    
    print(f'\nHighest confusion pairs:')
    for a, b, total in pairs[:5]:
        print(f'  {a} ↔ {b}: {total} verbs')
else:
    print('No confusion data available.')

## Test 4: Clustering (k-sweep)

In [ ]:
# ═══ Run k-means clustering if embeddings are available ═══
try:
    verb_embeddings
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    
    print('Running k-sweep (k=3 to k=15)...')
    n_sample = min(5000, len(verb_embeddings))
    
    results = []
    for k in range(3, 16):
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(verb_embeddings)
        sil = silhouette_score(verb_embeddings, labels, metric='cosine', sample_size=n_sample)
        results.append((k, sil, km.inertia_))
        marker = ' ◄◄◄' if k == 9 else ''
        print(f'  k={k:2d}  silhouette={sil:.4f}  inertia={km.inertia_:.0f}{marker}')
    
    best_k = max(results, key=lambda x: x[1])[0]
    print(f'\nBest k by silhouette: {best_k}')
    k9_sil = next(r[1] for r in results if r[0] == 9)
    best_sil = max(r[1] for r in results)
    print(f'k=9 silhouette: {k9_sil:.4f} ({k9_sil/best_sil*100:.1f}% of best)')
except NameError:
    print('No embeddings available. Generate them in 02_Embeddings first.')
    print('\nIf you have pre-computed data, the clustering results are in the metrics file.')

## Test 5: Boundary Analysis

In [ ]:
# ═══ Boundary verbs (most ambiguous assignments) ═══
if boundaries:
    ambig = boundaries.get('most_ambiguous', [])
    busiest = boundaries.get('busiest_boundaries', {})
    
    print(f'Mean gap: {boundaries.get("mean_gap", 0):.4f}')
    print(f'Median gap: {boundaries.get("median_gap", 0):.4f}')
    
    print(f'\nMost ambiguous verbs (smallest gap between top-2 operators):')
    print(f'{"Verb":>20s} {"Op1":>5s} {"Sim1":>7s} {"Op2":>5s} {"Sim2":>7s} {"Gap":>7s}')
    print('-' * 55)
    for v in ambig[:20]:
        print(f'{v["verb"]:>20s} {v["op1"]:>5s} {v["sim1"]:>7.4f} {v["op2"]:>5s} {v["sim2"]:>7.4f} {v["gap"]:>7.4f}')
    
    if busiest:
        print(f'\nBusiest operator boundaries:')
        for pair, count in sorted(busiest.items(), key=lambda x: -x[1])[:10]:
            print(f'  {pair}: {count} contested verbs')
else:
    print('No boundary data available.')

In [ ]:
# ═══ Download analysis results ═══
results = {
    'metrics': metrics,
    'boundaries': boundaries,
    'confusion': confusion,
}
# download_json(results, 'eo_analysis_results.json')
print('Uncomment download_json() above to save results.')